In [36]:
import pandas as pd
import os
from pathlib import Path

ENCONDING_IN = 'cp1252'

EMENDAS_CSV = Path(r'C:\Users\gbarb\Documents\PITON\projeto\database\EmendasParlamentares.csv')
CONVENIOS_CSV = Path(r'C:\Users\gbarb\Documents\PITON\projeto\database\EmendasParlamentares_Convenios.csv')
FAVORECIDOS_CSV = Path(r'C:\Users\gbarb\Documents\PITON\projeto\database\EmendasParlamentares_PorFavorecido.csv')

emendas = pd.read_csv(EMENDAS_CSV, delimiter=';', encoding=ENCONDING_IN)
convenios = pd.read_csv(CONVENIOS_CSV, delimiter=';', encoding=ENCONDING_IN)
favorecidos = pd.read_csv(FAVORECIDOS_CSV, delimiter=';', encoding=ENCONDING_IN)

df_emenda = emendas
df_convenio = convenios
df_favorecidos = favorecidos

C:\Users\gbarb\AppData\Local\Temp\ipykernel_17284\3808009729.py:11: DtypeWarning: Columns (0: Código da Emenda, 1: Código do Autor da Emenda, 2: Número da emenda) have mixed types. Specify dtype option on import or set low_memory=False.
  emendas = pd.read_csv(EMENDAS_CSV, delimiter=';', encoding=ENCONDING_IN)
C:\Users\gbarb\AppData\Local\Temp\ipykernel_17284\3808009729.py:12: DtypeWarning: Columns (0: Código da Emenda) have mixed types. Specify dtype option on import or set low_memory=False.
  convenios = pd.read_csv(CONVENIOS_CSV, delimiter=';', encoding=ENCONDING_IN)


In [37]:
#df_emenda.info()

In [38]:
#df_convenio.info()

In [39]:
#df_favorecidos.info()

CRIAÇÃO CATALOGO E DATALAKE (DATABRICKS)

In [40]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

w = WorkspaceClient(profile="DATALAKE_EXs")

CATALOG = "datalake_emendas"

for schema in ["bronze", "silver", "gold"]:
    try:
        w.schemas.create(
            name=schema,
            catalog_name=CATALOG,
            comment=f"Schema {schema}"
        )
        print(f"[OK] {CATALOG}.{schema}")
    except ResourceAlreadyExists:
        print(f'[INFO] Schema lá existe: {CATALOG}.{schema}')
    except Exception as e:
        print(f"[ERRO] {CATALOG}.{schema}: {e}")


[ERRO] datalake_emendas.bronze: Schema 'bronze' already exists
[ERRO] datalake_emendas.silver: Schema 'silver' already exists
[ERRO] datalake_emendas.gold: Schema 'gold' already exists


In [ ]:
from IPython.core import async_helpers
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists
from databricks.sdk.service.catalog import VolumeType

w = WorkspaceClient(profile="DATALAKE_EXs")

CATALOG = "datalake_emendas"
SCHEMA = "bronze"
VOLUME_NAME = "raw"

# Caminho completo do volume dentro do Databricks
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

# 1. Criar o Volume (se não existir)
try:
    w.volumes.create(
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        name=VOLUME_NAME,
        volume_type=VolumeType.MANAGED,
        comment="Volume para receber arquivos brutos"
    )
    print(f"[OK] Volume criado: {volume_path}")
except ResourceAlreadyExists:
    print(f"[INFO] Volume já existe: {volume_path}")

# 2. Upload de todos os arquivos de uma pasta

pasta_database = r'C:\Users\gbarb\Documents\PITON\projeto\database'

# Lista todos os arquivos dentro da pasta
print("Iniciando o upload dos arquivos...")

for arq in os.listdir(pasta_database):
        # Verifica se o arquivo é um CSV (ignora arquivos ocultos ou outras extensões)
    if arq.endswith('.csv'):
                # Monta o caminho completo no PC e o destino no Databricks
        caminho_arquivo_local = os.path.join(pasta_database, arq)
        caminho_no_databricks = f"{volume_path}/{arq}"
        print(f"-> Enviando: {arq} ...")

# Abre o arquivo em modo leitura binária ("rb") e envia

        with open(caminho_arquivo_local, "rb") as f:
            w.files.upload(caminho_no_databricks, f)
    
print("[OK] Upload concluído com sucesso!")


[INFO] Volume já existe: /Volumes/datalake_emendas/bronze/raw
Iniciando o upload dos arquivos...
-> Enviando: EmendasParlamentares.csv ...
